# centennial bluff mission a


## 1. Organize data
Create a folder under `semantic_SfM/data` and organize your data following the structures below. 

Agisoft:
```
semantic_SfM/data
    ├── centennial_bluff/mission_b
        ├── DJI_photos
        │       ├── DJI_0000.JPG
        │       ├── DJI_0001.JPG
        │       ├── ...
        │       └── DJI_0999.JPG
        ├── SfM_products
        │       ├── a.xml
        │       ├── a.jpg
        │       ├── a.mtl
        │       ├── a.obj
        │       ├── a.las       
        │       └── a_downsampled.las   
        ├── segmentations
        └── associations

```

In [1]:
import os

scene_dir = '../data/centennial_bluff/mission_b'
pointcloud_path = os.path.join(scene_dir, 'SfM_products', 'b_downsampled.las')
associations_folder_path = os.path.join(scene_dir, 'associations')
segmentations_folder_path = os.path.join(scene_dir, 'segmentations')
photos_folder_path = os.path.join(scene_dir, 'DJI_photos')
camera_path = os.path.join(scene_dir, 'SfM_products', 'b.xml')
mesh_path = os.path.join(scene_dir, 'SfM_products', 'b.obj')

## 2. Create 2D Segmentation using SAM

In [2]:
from ssfm.image_segmentation import ImageSegmentation
import os

In [3]:
sam_params = {}
sam_params['model_name'] = 'sam2'
sam_params['model_path'] = '../semantic_SfM/sam2/sam2.1_hiera_large.pt'
sam_params['device'] = 'cuda:2'
sam_params['points_per_side'] = 64
sam_params['points_per_batch'] = 128
sam_params['pred_iou_thresh'] = 0.6
sam_params['stability_score_offset'] = 0.5
sam_params['box_nms_thresh'] = 0.6
sam_params['use_m2m'] = True
sam_params['crop_n_layers'] = 3


image_path_list = [os.path.join(photos_folder_path, image) for image in os.listdir(photos_folder_path)]

# sort images based on the values of keyimages in file names
image_path_list = sorted(image_path_list, key=lambda x: int(x.split('/')[-1].split('.')[0].split('_')[-1]))

image_list = [image for image in os.listdir(photos_folder_path)]

# sort images based on the values of keyimages in file names
image_list = sorted(image_list, key=lambda x: int(x.split('/')[-1].split('.')[0].split('_')[-1]))

# print the length of the image list
print(len(image_list))

632


In [4]:
run_segmentation = False

if run_segmentation:
    image_segmentor = ImageSegmentation(sam_params)   
    image_segmentor.set_distortion_correction(camera_path)
    image_segmentor.batch_predict(image_path_list, segmentations_folder_path, save_overlap=True, skip_existing=False)

## 3. Create projection associations

In [5]:
from ssfm.probabilistic_projection import *
import time

In [6]:
pointcloud_projector = PointcloudProjection(depth_filtering_threshold=0.01, effective_depth = np.inf)

In [7]:
pointcloud_projector.read_camera_parameters(camera_path)
pointcloud_projector.read_mesh(mesh_path)
pointcloud_projector.read_pointcloud(pointcloud_path)

In [8]:
pointcloud_projector.parallel_batch_project_joblib(image_list, associations_folder_path, num_workers=8, save_depth=True)

Processing frames: 100%|██████████| 632/632 [1:02:41<00:00,  5.95s/it]


In [9]:
# simple segmentation filter
from ssfm.simple_mask_filter import SimpleMaskFilter

In [10]:
segmentation_folder_path = "../data/centennial_bluff/mission_b/segmentations"

configs = {
    'window_size': 5,
    'depth_folder': os.path.join(associations_folder_path, 'depth'),
    'output_folder': "../data/centennial_bluff/mission_b/segmentations_filtered",
    'area_upper_threshold': 25,
    'area_lower_threshold': 0.01,
    'erosion_kernel_size': 5,
    'erosion_iteration':1,
    'camera_parameter_file': "../data/centennial_bluff/mission_b/SfM_products/b.xml",
    'background_mask': True
}


mask_filter = SimpleMaskFilter(configs)

#mask_filter.filter_segmentation_file('../../data/courtright/segmentations/DJI_0650.npy')
mask_filter.filter_batch_processes(segmentation_folder_path, num_processes=16)

Total number of files: 632


100%|██████████| 632/632 [58:02<00:00,  5.51s/it]


In [5]:
segmentations_folder_path = "../data/centennial_bluff/mission_b/segmentations_filtered"

In [7]:
# build keyimage associations
from ssfm.keyimage_associations_builder import *

In [13]:
smc_solver = KeyimageAssociationsBuilder(image_list, associations_folder_path, segmentations_folder_path)

In [14]:
smc_solver.build_associations()

100%|██████████| 632/632 [02:57<00:00,  3.55it/s]


In [15]:
smc_solver.find_min_cover()

| Metric                                                       | Count      | Percentage           |
----------------------------------------------------------------------------------------------------
| Number of points not covered by any image                    | 8556702    | 40.46                |
| Number of points covered by less than or equal to 1 image    | 8845330    | 41.83                |
| Number of points covered by less than or equal to 3 images   | 9175316    | 43.39                |
| Number of points covered by less than or equal to 5 images   | 9452688    | 44.70                |


## 4. Estimate memory usage

In [16]:
from ssfm.memory_calculator import memory_calculator

In [17]:
# pointcloud file
las_file = pointcloud_path
# image file sample; this needs to be an original image even if patch images are used
image_file = os.path.join(photos_folder_path, image_list[0])
# number of images
num_images = len(image_list)
# number of segmentation ids for each point in the point cloud
num_segmentation_ids = 5

memory_calculator(las_file, image_file, num_images, num_segmentation_ids)

+----------------------------------------+----------------------+
|              Memory Type               | Memory Required (GB) |
+----------------------------------------+----------------------+
|      Segmentation for each image       | 0.037181854248046875 |
| Pixel2point association for each image | 0.07436370849609375  |
| Point2pixel association for each image | 0.07878156751394272  |
|                                        |                      |
|      Segmentation for all images       |  23.498931884765625  |
| Pixel2point association for all images |  46.99786376953125   |
| Point2pixel association for all images |   49.7899506688118   |
|          pc_segmentation_ids           |  0.3939078375697136  |
|         pc_segmentation_probs          |  0.3939078375697136  |
|          keyimage_association          |  12.44748766720295   |
|                 Total                  |  133.52204966545105  |
+----------------------------------------+----------------------+


## 5. Run object registration

In [6]:
from ssfm.object_registration import *
from ssfm.post_processing import *
import time

In [12]:
obr = ObjectRegistration(pointcloud_path, segmentations_folder_path, associations_folder_path, image_list=image_list, using_graph=False, radius=2, decaying=1, scene_name='mission_b')

# Run object registration
obr.object_registration(iou_threshold=0.5, save_semantics=True)

Processing images:  43%|████▎     | 271/632 [38:29:03<51:15:54, 511.23s/it]


KeyboardInterrupt: 

In [7]:
image_id = 270
semantics_folder_path = os.path.join(associations_folder_path, 'semantics', 'semantics_{}.npy'.format(image_id))
save_las_path = os.path.join(associations_folder_path, 'semantics', 'semantics_{}.las'.format(image_id))
add_semantics_to_pointcloud(pointcloud_path, semantics_folder_path, save_las_path, remove_small_N=50, nearest_interpolation=50)

Before removing small semantics: 
maximum of semantics:  100184
number of unique semantics:  13791
After removing small semantics: 
number of unique semantics:  2748


In [8]:
semantic_pc_file_path = save_las_path
post_processing = PostProcessing(semantic_pc_file_path)
post_processing.shuffle_semantic_ids(exclude_largest_semantic=False)
save_las_path = os.path.join(associations_folder_path, 'semantics', 'semantics_{}_shuffled.las'.format(image_id))
post_processing.save_semantic_pointcloud(save_las_path)

Number of unique semantics:  2740


In [9]:
semantic_pc_file_path = save_las_path
post_processing = PostProcessing(semantic_pc_file_path)
post_processing.sort_semantic_ids(exclude_largest_semantic=False)
save_las_path = os.path.join(associations_folder_path, 'semantics', 'semantics_{}_sorted.las'.format(image_id))
post_processing.save_semantic_pointcloud(save_las_path)

Number of unique semantics:  2740
